In [ ]:
from pathlib import Path
import importlib.util
import json
import shutil
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone

import numpy as np
import pandas as pd

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
DATA = WORK / "cbs-data"
PROJECT_SOURCE_CANDIDATES = [
    INPUT / "notebooks/urajkaushik/cosmic-swan-4/cbs-project-src",
    INPUT / "datasets/urajkaushik/cosmic-swan-4/cbs-project-src",
]
DATA_SOURCE_CANDIDATES = [
    INPUT / "notebooks/urajkaushik/cosmic-swan-4/cbs-data",
    INPUT / "datasets/urajkaushik/cosmic-swan-4/cbs-data",
]

PROJECT_SOURCE = next(
    (p for p in PROJECT_SOURCE_CANDIDATES if (p / "pyproject.toml").exists()), None
)
if PROJECT_SOURCE is None:
    for p in INPUT.rglob("pyproject.toml"):
        candidate = p.parent
        if (candidate / "src/cosmic_black_swan/cli.py").exists():
            PROJECT_SOURCE = candidate
            break
if PROJECT_SOURCE is None:
    raise FileNotFoundError(
        "Attach the Cosmic Swan 4 notebook output; cbs-project-src/pyproject.toml was not found."
    )

data_candidates = [p for p in DATA_SOURCE_CANDIDATES if all(
    (p / name).exists() for name in ("manifest.csv", "features.csv", "series.npy")
)]
if not data_candidates:
    for p in INPUT.rglob("manifest.csv"):
        folder = p.parent
        if all((folder / name).exists() for name in ("features.csv", "series.npy")):
            data_candidates.append(folder)
if not data_candidates:
    raise FileNotFoundError(
        "Attach the Cosmic Swan 4 output containing cbs-data/manifest.csv, features.csv, and series.npy."
    )

def manifest_count(folder):
    frame = pd.read_csv(folder / "manifest.csv", usecols=["status"])
    return int(frame["status"].astype(str).isin(["COMPLETE", "ARCHIVED"]).sum())

DATA_SOURCE = max(data_candidates, key=manifest_count)
PROJECT = WORK / "cbs-project-src"
PROJECT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(PROJECT_SOURCE, PROJECT, dirs_exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)
for name in ("manifest.csv", "features.csv", "series.npy"):
    destination = DATA / name
    if not destination.exists():
        shutil.copy2(DATA_SOURCE / name, destination)
for folder_name in ("scores", "pixel_audits"):
    source_folder = DATA_SOURCE / folder_name
    dest_folder = DATA / folder_name
    if source_folder.exists():
        shutil.copytree(source_folder, dest_folder, dirs_exist_ok=True)

# The attached raw FITS are read-only and the 5,000 rows already have matching features/sequences.
# Mark those archived in the writable manifest so featurize(append=True) processes only new files.
manifest_path = DATA / "manifest.csv"
manifest = pd.read_csv(manifest_path, dtype={"source_id": "string"})
if "local_path" in manifest:
    completed = manifest["status"].astype(str).eq("COMPLETE")
    local_paths = manifest["local_path"].fillna("").astype(str)
    missing_old = completed & ~local_paths.map(lambda value: bool(value) and Path(value).exists())
    manifest.loc[missing_old, "status"] = "ARCHIVED"
    manifest.loc[missing_old, "local_path"] = ""
    if "message" in manifest:
        manifest.loc[missing_old, "message"] = "Raw FITS retained in attached read-only input; features and series are present."
    manifest.to_csv(manifest_path, index=False)

print("Project source:", PROJECT_SOURCE)
print("Writable project copy:", PROJECT)
print("Seed data:", DATA_SOURCE)
print("Working data root:", DATA)
print("Seed completed manifest products:", manifest_count(DATA_SOURCE))
print("Seed feature rows:", len(pd.read_csv(DATA / "features.csv")))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)],
    check=True,
)
sys.path.insert(0, str(PROJECT / "src"))
importlib.invalidate_caches()


In [ ]:
# Shared stage recorder: a failed optional/network stage is visible but does not hide later outputs.
STAGE_LOG = []
STATE_FILE = DATA / "day6_run_state.json"

def run_stage(name, function):
    started = time.time()
    print(f"\n===== {name} =====", flush=True)
    try:
        result = function()
        row = {"stage": name, "status": "complete", "seconds": round(time.time() - started, 1)}
        if isinstance(result, dict):
            row["result"] = result
        print(json.dumps(row, indent=2, default=str), flush=True)
    except Exception as exc:
        row = {
            "stage": name,
            "status": "failed",
            "seconds": round(time.time() - started, 1),
            "error": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc(limit=8),
        }
        print(row["error"], flush=True)
    STAGE_LOG.append(row)
    STATE_FILE.write_text(json.dumps(STAGE_LOG, indent=2, default=str), encoding="utf-8")
    return row

def get_inventory():
    manifest = pd.read_csv(DATA / "manifest.csv", dtype={"source_id": "string"})
    features = pd.read_csv(DATA / "features.csv", dtype={"source_id": "string"})
    usable_status = manifest["status"].astype(str).isin(["COMPLETE", "ARCHIVED"])
    tics = features["source_id"].astype("string")
    valid_tics = tics[tics.str.fullmatch(r"\d{8,16}", na=False)]
    return {
        "completed_or_archived_products": int(usable_status.sum()),
        "feature_observations": int(len(features)),
        "unique_tic_sources_with_features": int(valid_tics.nunique()),
        "feature_failures": int(pd.read_csv(DATA / "feature_errors.csv").shape[0])
        if (DATA / "feature_errors.csv").exists() else 0,
    }


In [ ]:
from cosmic_black_swan.fetch import fetch_tess
from cosmic_black_swan.features import featurize
from cosmic_black_swan.score import score_isolation_forest

TARGET_PRODUCTS = 20_000

def fetch_to_target():
    before = get_inventory()
    if before["completed_or_archived_products"] >= TARGET_PRODUCTS:
        return {"action": "already_at_target", **before}
    result = fetch_tess(
        data_root=DATA,
        sectors=list(range(1, 97)),
        max_lightcurves=TARGET_PRODUCTS,
        per_sector_cap=300,
        seed=42,
        batch_size=32,
        pause_seconds=0.05,
    )
    return {"fetch_result": result, "before": before, "after": get_inventory()}

run_stage("fetch toward 20,000 cumulative TESS products", fetch_to_target)
run_stage("append features for newly downloaded light curves", lambda: featurize(DATA, append=True))
run_stage("refit Isolation Forest on the available feature table", lambda: score_isolation_forest(DATA, contamination=0.02, seed=42))
print("Inventory after fetch/features/scoring:", json.dumps(get_inventory(), indent=2))


In [ ]:
from cosmic_black_swan.catalog_labels import build_catalog_labels
from cosmic_black_swan.classifier import train_xgboost
from cosmic_black_swan.conformal import run_conformal_screen
from cosmic_black_swan.injections import injection_recovery
from cosmic_black_swan.transformer import train_transformer

LABELS = DATA / "labels.csv"
run_stage("cross-match supported Gaia/TESS catalog labels", lambda: build_catalog_labels(DATA, match_radius_arcsec=2.0))

def fit_xgboost_if_supported():
    if not LABELS.exists():
        return {"status": "skipped", "reason": "No labels.csv was produced by the catalog cross-match."}
    labels = pd.read_csv(LABELS)
    if not {"source_id", "class_label"}.issubset(labels.columns):
        return {"status": "skipped", "reason": "Catalog labels lack source_id/class_label."}
    if labels["class_label"].dropna().nunique() < 2 or labels["source_id"].nunique() < 10:
        return {"status": "skipped", "reason": "Not enough supported catalog-labeled sources/classes for a held-out model."}
    return train_xgboost(DATA, LABELS, seed=42, minimum_evidence="single_catalog")

run_stage("source-disjoint XGBoost catalog-class baseline", fit_xgboost_if_supported)

def fit_transformer():
    if importlib.util.find_spec("torch") is None:
        return {"status": "skipped", "reason": "PyTorch is unavailable in this Kaggle runtime."}
    return train_transformer(DATA, epochs=5, batch_size=128, learning_rate=3e-4, seed=42)

run_stage("masked-patch Transformer held-out source scoring", fit_transformer)
run_stage("source-disjoint field-relative conformal ranking", lambda: run_conformal_screen(DATA, seed=42))
run_stage("held-out synthetic injection recovery and detector-location map", lambda: injection_recovery(DATA, sample_size=500, amplitudes_sigma=(1, 2, 4, 8), seed=42))

conformal_record = DATA / "scores/conformal_run.json"
if conformal_record.exists():
    conformal = json.loads(conformal_record.read_text(encoding="utf-8"))
    print("Conformal interpretation:", conformal.get("interpretation"))
    print("FDR control claimed by this implementation:", conformal.get("fdr_control_claim"))


In [ ]:
# Build a concise evidence report from files that actually exist after this Run All.
inventory = get_inventory()
score_dir = DATA / "scores"
report = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "target_completed_products": TARGET_PRODUCTS,
    "inventory": inventory,
    "target_reached": inventory["completed_or_archived_products"] >= TARGET_PRODUCTS,
    "stages": STAGE_LOG,
    "artifact_paths": {
        name: str(path) for name, path in {
            "manifest": DATA / "manifest.csv",
            "features": DATA / "features.csv",
            "series": DATA / "series.npy",
            "isolation_forest_scores": score_dir / "isolation_forest.csv",
            "xgboost_run": score_dir / "xgboost_run.json",
            "transformer_run": score_dir / "transformer_run.json",
            "conformal_run": score_dir / "conformal_run.json",
            "injection_run": score_dir / "injection_recovery_run.json",
        }.items() if path.exists()
    },
    "claim_limits": {
        "fdr_control": "Not established: the current conformal calibration sample is drawn from the observed field, not an independently vetted nominal population.",
        "catalog_unmatched": "Unmatched sources remain unlabeled; they are not treated as normal examples.",
        "score_semantics": "Anomaly percentiles and model scores are screening ranks, not probabilities or discoveries.",
    },
}
report_path = WORK / "day6_run_summary.json"
report_path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
print("Saved:", report_path)
print(json.dumps({"inventory": inventory, "target_reached": report["target_reached"], "stages": [{"stage": s["stage"], "status": s["status"]} for s in STAGE_LOG], "artifacts": report["artifact_paths"]}, indent=2))
